# Compare GrooD markers vs. cell marker

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
import math
warnings.filterwarnings('ignore')
import joblib
import matplotlib.colors as mcolors

# Load cell markers

In [2]:
df = pd.read_excel('Cell_marker_2.0_all_human_markers.xlsx')
df

,species,tissue_class,tissue_type,uberonongology_id,cancer_type,cell_type,cell_name,cellontology_id,marker,Symbol,GeneID,Genetype,Genename,UNIPROTID,technology_seq,marker_source,PMID,Title,journal,year
0,Human,Abdomen,Abdomen,UBERON_0000916,Normal,Normal cell,Macrophage,CL_0000235,MERTK,MERTK,10461.0,protein_coding,"MER proto-oncogene, tyrosine kinase",Q12866,NaN,Experiment,31982413.0,Peritoneal Level of CD206 Associates With Mort...,Gastroenterology,2020.0
1,Human,Abdomen,Abdomen,UBERON_0000916,Normal,Normal cell,Macrophage,CL_0000235,CD16,FCGR3A,2215.0,protein_coding,Fc fragment of IgG receptor IIIb,O75015,NaN,Experiment,31982413.0,Peritoneal Level of CD206 Associates With Mort...,Gastroenterology,2020.0
2,Human,Abdomen,Abdomen,UBERON_0000916,Normal,Normal cell,Macrophage,CL_0000235,CD206,MRC1,4360.0,protein_coding,mannose receptor C-type 1,P22897,NaN,Experiment,31982413.0,Peritoneal Level of CD206 Associates With Mort...,Gastroenterology,2020.0
3,Human,Abdomen,Abdomen,UBERON_0000916,Normal,Normal cell,Macrophage,CL_0000235,CRIg,VSIG4,11326.0,protein_coding,V-set and immunoglobulin domain containing 4,Q9Y279,NaN,Experiment,31982413.0,Peritoneal Level of CD206 Associates With Mort...,Gastroenterology,2020.0
4,Human,Abdomen,Abdomen,UBERON_0000916,Normal,Normal cell,Macrophage,CL_0000235,CD163,CD163,9332.0,protein_coding,CD163 molecule,Q86VB7,NaN,Experiment,31982413.0,Peritoneal Level of CD206 Associates With Mort...,Gastroenterology,2020.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60872,Human,Vein,Vein,UBERON_0001638,Normal,Normal cell,Venous cell,NaN,LRG1,LRG1,116844.0,protein_coding,leucine rich alpha-2-glycoprotein 1,P02750,10x Chromium,Experiment,33996252.0,Single-cell transcriptomics reveals the landsc...,Molecular therapy. Nucleic acids,2021.0
60873,Human,Vein,Vein,UBERON_0001638,Normal,Normal cell,Venous cell,NaN,EMCN,EMCN,51705.0,protein_coding,endomucin,Q9ULC0,10x Chromium,Experiment,33996252.0,Single-cell transcriptomics reveals the landsc...,Molecular therapy. Nucleic acids,2021.0
60874,Human,Vein,Vein,UBERON_0001638,Normal,Normal cell,Systemic–venous endothelial cell,NaN,COL15A1,COL15A1,1306.0,protein_coding,collagen type XV alpha 1 chain,B3KTP7,10x Chromium,Experiment,34030460.0,Integrated Single-Cell Atlas of Endothelial Ce...,Circulation,2021.0
60875,Human,NaN,Intestine,UBERON_0000160,Normal,Normal cell,B cell,CL_0000236,CD19,CD19,930.0,protein_coding,CD19 molecule,P15391,10x Chromium,Experiment,34269788.0,CD16+CD163+ monocytes traffic to sites of infl...,The Journal of experimental medicine,2021.0


In [6]:
list(set(df['cell_name'].tolist()))

['Breast cancer stem-like cell',
 'Mononuclear blast cell',
 'Innate B cell',
 'Small intestinal stem cell',
 'H1 embryonic stem cell',
 'Immature endothelial cell',
 'CD34+ hematopoietic stem cell',
 'Endoderm progenitor cell',
 'Primordial germ cell',
 'Cancer-initiating cell',
 'Common lymphoid progenitor cell',
 'Bone marrow stromal cell',
 'Epidermal stem cell',
 'Peripheral immune cell',
 'Tuft progenitor cell',
 'Immune oligodendroglial cell(imOLG)',
 'Class switched memory B IgGκ cell',
 'Skeletal muscle cell',
 'Lower motor neuron',
 'Perivascular-like cell',
 'Mullerian duct epithelial cell',
 'Smooth muscle cell',
 'IgA+ Regulatory B cell',
 'Spermatogonial stem cell\xa0',
 'Hematopoietic progenitor cell',
 'Loop of Henle cell',
 'Hepatic stellate cell',
 'Myoepithelial cell',
 'Adult neural stem cell',
 'PD-L1+ Regulatory B cell',
 'Vas afferen cell',
 'Melanocyte',
 'Induced pluripotent stem cell (iPSC)',
 'Proximal tubule (PT) cell',
 'Amniotic fluid stem cell',
 'Platele

In [30]:
model = joblib.load('../../grood_runs/GrooD_Hao_pseudobulk_new/intersect_CPM/train/model/Model.pkl')
model

{'metadata': {'estimators': ['B cells',
   'CD4 T cells',
   'CD8 T cells',
   'DC',
   'Monocytes',
   'NK cells',
   'Tregs'],
  'model_type': 'grood',
  'norm': 'CPM'},
 'model': MultiOutputRegressor(estimator=GradientBoostingRegressor(learning_rate=0.01,
                                                          max_depth=4,
                                                          min_samples_split=50,
                                                          n_estimators=500),
                      n_jobs=8)}

In [35]:
# Plotting feature importances
def plot_marker_expression_correlation(model, estimator):
    
    # Get list of cellTypes (same order as in model estimators)
    genes = model['model'].estimators_[0].feature_names_in_.tolist()

    index = model['metadata']['estimators'].index(estimator)

    # iterate over n or cellTypes

    feature_importance = model['model'].estimators_[index].feature_importances_
    sorted_idx = np.argsort(feature_importance) # sort highest to lowest importance and get the feature indices
    most_important_features = sorted_idx[-10:] # get 20 most important features
    marker_genes = np.array(genes)[most_important_features]

    return list(marker_genes)

grood_markers_dc = plot_marker_expression_correlation(model, 'DC')
grood_markers_dc

['CLEC10A',
 'GAS6',
 'MAT2A',
 'PLD4',
 'CYP2S1',
 'HLA-DPA1',
 'TAGLN2',
 'FLT3',
 'FCER1A',
 'SPINT2']

In [36]:
marker_dc = list(set(df.loc[df['cell_name'] == 'Dendritic cell'].loc[df['tissue_class'] == 'Blood', 'marker'].tolist()))
marker_dc

['CD86',
 'HSPA5',
 'CD123',
 'MZB1',
 'IRF7',
 'HLA-DQA1',
 'IL3RA',
 'IGJ',
 'FLT3',
 'CD141',
 'LILRA4',
 'UBC',
 'CCR7',
 'CD80',
 'CD163',
 'CD83',
 'CLEC4C',
 'CD40',
 'MS4A3',
 'FCER1A',
 'IL5RA',
 'NRP1',
 'ISG15',
 'IFI6',
 'HLA-DPA1',
 'IFITM3',
 'IRF8',
 'PLD4',
 'CD1E',
 'CLEC10A',
 'HLA-DR',
 'CD1c',
 'Lin',
 'CST3',
 'HLADR',
 'CLEC12A',
 'LILRB4',
 'CD11c',
 'LYZ',
 'Sox2',
 'CYBA',
 'CD34',
 'MHC class II',
 'CLEC9A',
 'CD1C',
 'Bmp4',
 'HPGD']

In [37]:
len(marker_dc), len(grood_markers_dc), len(list(set(set(marker_dc) - set(grood_markers_dc))))

(47, 10, 42)

In [38]:
list(set(set(grood_markers_dc) - set(marker_dc)))

['TAGLN2', 'CYP2S1', 'GAS6', 'MAT2A', 'SPINT2']